# Character-Level Text Generation with RNN

## Objectives
Train a **char-RNN** to generate Shakespeare-like text character by character.

## Theory
Each character is encoded; the LSTM predicts the **next character** via softmax over the vocabulary. **Temperature** scales randomness at generation time (higher = more creative, lower = more conservative).


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Generative RNNs underpin autocomplete, marketing copy drafts, and chatbots (modern systems use Transformers, but char-RNNs teach sequence fundamentals).


In [ ]:
import requests
url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
text = requests.get(url, timeout=30).text
print("Characters:", len(text))
print(text[:200])


In [ ]:
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
vocab_size = len(chars)

seq_length = 100
step = 3
sequences = []
next_chars = []
for i in range(0, len(text) - seq_length, step):
    sequences.append(text[i:i+seq_length])
    next_chars.append(text[i+seq_length])

x = np.array([[char2idx[c] for c in seq] for seq in sequences])
y = np.array([char2idx[c] for c in next_chars])
print(x.shape, y.shape)


In [ ]:
# One-hot encode for training (subset for speed)
subset = 20000
x_ohe = keras.utils.to_categorical(x[:subset], num_classes=vocab_size)
y_ohe = keras.utils.to_categorical(y[:subset], num_classes=vocab_size)
x_train, x_val, y_train, y_val = train_test_split(x_ohe, y_ohe, test_size=0.1, random_state=SEED)
print("Train sequences:", x_train.shape)


In [ ]:
model = models.Sequential([
    layers.LSTM(128, input_shape=(seq_length, vocab_size)),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
char_cb = [
    callbacks.ModelCheckpoint("char_rnn_best.keras", save_best_only=True),
    callbacks.EarlyStopping(patience=3, restore_best_weights=True),
]
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=20, batch_size=64,
          callbacks=char_cb, verbose=1)
pd.DataFrame(history.history).plot()
plt.show()


In [ ]:
def generate(seed, length=200, temperature=1.0):
    out = seed
    for _ in range(length):
        x_pred = np.zeros((1, seq_length, vocab_size))
        seq = out[-seq_length:]
        for t, ch in enumerate(seq):
            x_pred[0, t, char2idx[ch]] = 1
        preds = model.predict(x_pred, verbose=0)[0]
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        idx = np.random.choice(len(preds), p=exp_preds / np.sum(exp_preds))
        out += idx2char[idx]
    return out

print(generate("ROMEO: ", length=300))
model.save("char_rnn_shakespeare.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
